In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pypfopt import EfficientFrontier, risk_models, expected_returns
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices
import pickle
import os

os.makedirs("../results/figures", exist_ok=True)
pd.set_option("display.float_format", "{:.3f}".format)

print("Librairies chargées ✓")

Librairies chargées ✓


In [3]:
# Chargement
with open("../results/gradient_boosting.pkl", "rb") as f:
    model = pickle.load(f)

dataset   = pd.read_csv("../data/dataset.csv", parse_dates=["date"])
prices    = pd.read_csv("../data/prices.csv",
                         index_col=0, parse_dates=True)
tickers   = pd.read_csv("../data/jse_tickers.csv")
allocation_df = pd.read_csv("../results/portfolio_allocation.csv")

FEATURES = [
    "trailingPE", "priceToBook", "debtToEquity",
    "returnOnEquity", "returnOnAssets", "profitMargins",
    "currentRatio", "floatShares", "momentum_1m",
    "momentum_3m", "momentum_6m", "volatility", "rel_strength",
]

print(f"Modèle chargé     : Gradient Boosting ✓")
print(f"Dataset           : {dataset.shape}")
print(f"Prix              : {prices.shape}")
print(f"\nAllocation finale :")
print(allocation_df)

Modèle chargé     : Gradient Boosting ✓
Dataset           : (24533, 16)
Prix              : (1249, 22)

Allocation finale :
   ticker  poids_%  nb_actions  prix_ZAR  montant_ZAR
0  GFI.JO   40.000           1 27754.490    27754.490
1  IMP.JO   27.400           3  9126.200    27378.590
2  REM.JO   13.800           1 16239.750    16239.750
3  SBK.JO    5.000           1 20781.220    20781.220
4  FSR.JO    5.000           1  7342.260     7342.260


## Optimisation du portefeuille JSE

Cette étape combine deux approches complémentaires :

**Étape 1 — Sélection ML** : le modèle identifie les actions les plus susceptibles de surperformer l'indice JSE parmi nos 22 titres.

**Étape 2 — Optimisation Markowitz** : parmi les actions retenues, on calcule la répartition optimale du capital pour maximiser le **ratio de Sharpe** — c'est à dire le meilleur rendement possible pour le niveau de risque pris.

L'idée clé : la sélection ML répond à *"quoi acheter ?"* et Markowitz répond à *"combien investir dans chaque action ?"*

In [4]:
# Snapshot à la dernière date
last_date = dataset["date"].max()
snapshot  = dataset[dataset["date"] == last_date].copy()
snapshot["proba"] = model.predict_proba(snapshot[FEATURES])[:, 1]
snapshot = snapshot.sort_values("proba", ascending=False)

# Toutes les actions avec leur score
fig = px.bar(
    snapshot,
    x="proba",
    y="ticker",
    orientation="h",
    title=f"Score ML par action — {last_date.date()}",
    labels={
        "proba" : "Probabilité de surperformance",
        "ticker": "Action"
    },
    color="proba",
    color_continuous_scale=["#D85A30", "#F5C4B3", "#1D9E75"],
    template="plotly_white",
    width=700, height=600
)

fig.add_vline(
    x=0.55,
    line_dash="dash",
    line_color="gray",
    annotation_text="Seuil de sélection (55%)"
)

fig.update_layout(coloraxis_showscale=False)
fig.show()

fig.write_html("../results/figures/05_scores_ml.html")
fig.write_image("../results/figures/05_scores_ml.png")

print(f"\nActions sélectionnées (proba ≥ 55%) :")
selected = snapshot[snapshot["proba"] >= 0.55][["ticker", "proba"]]
print(selected.to_string(index=False))


Actions sélectionnées (proba ≥ 55%) :
ticker  proba
REM.JO  0.837
IMP.JO  0.754
AGL.JO  0.715
SBK.JO  0.713
GFI.JO  0.703
FSR.JO  0.685


In [5]:
# Actions sélectionnées par le modèle
selected_tickers = snapshot[
    snapshot["proba"] >= 0.55
]["ticker"].tolist()

prices_selected = prices[selected_tickers].dropna()

# Calcul des rendements espérés et covariance
mu = expected_returns.mean_historical_return(
    prices_selected, frequency=252
)
S  = risk_models.sample_cov(prices_selected, frequency=252)

# Simulation de 3000 portefeuilles aléatoires
np.random.seed(42)
n_portfolios = 3000
n_assets     = len(selected_tickers)

port_returns  = []
port_vols     = []
port_sharpes  = []

for _ in range(n_portfolios):
    w = np.random.dirichlet(np.ones(n_assets))
    r = float(np.dot(w, mu))
    v = float(np.sqrt(np.dot(w, np.dot(S.values, w))))
    s = (r - 0.08) / v
    port_returns.append(r)
    port_vols.append(v)
    port_sharpes.append(s)

portfolios_df = pd.DataFrame({
    "rendement"  : port_returns,
    "volatilite" : port_vols,
    "sharpe"     : port_sharpes
})

# Portefeuille optimal (max Sharpe)
ef = EfficientFrontier(mu, S, weight_bounds=(0.05, 0.40))
ef.max_sharpe(risk_free_rate=0.08)
perf = ef.portfolio_performance(
    verbose=False, risk_free_rate=0.08
)

fig = px.scatter(
    portfolios_df,
    x="volatilite",
    y="rendement",
    color="sharpe",
    title="Frontière efficiente — Portefeuilles JSE simulés",
    labels={
        "volatilite" : "Volatilité (risque)",
        "rendement"  : "Rendement espéré",
        "sharpe"     : "Sharpe ratio"
    },
    color_continuous_scale=["#D85A30", "#F5C4B3", "#1D9E75"],
    opacity=0.5,
    template="plotly_white",
    width=750, height=550
)

# Portefeuille optimal
fig.add_trace(go.Scatter(
    x=[perf[1]], y=[perf[0]],
    mode="markers",
    marker=dict(
        color="#534AB7",
        size=15,
        symbol="star"
    ),
    name=f"Optimal (Sharpe={perf[2]:.3f})"
))

fig.update_layout(legend=dict(x=0.7, y=0.1))
fig.show()

fig.write_html("../results/figures/05_frontiere_efficiente.html")
fig.write_image("../results/figures/05_frontiere_efficiente.png")

In [6]:
fig = px.pie(
    allocation_df,
    values="poids_%",
    names="ticker",
    title="Allocation optimale du portefeuille JSE",
    color_discrete_sequence=[
        "#1D9E75", "#534AB7", "#D85A30",
        "#BA7517", "#185FA5", "#888780"
    ],
    template="plotly_white",
    width=600, height=500
)

fig.update_traces(
    textposition="inside",
    textinfo="percent+label",
    hole=0.3
)

fig.show()

fig.write_html("../results/figures/05_allocation_portefeuille.html")
fig.write_image("../results/figures/05_allocation_portefeuille.png")

In [7]:
# Tableau propre avec toutes les infos
display_df = allocation_df.copy()
display_df = display_df.merge(
    tickers[["ticker", "name"]], on="ticker", how="left"
)

display_df = display_df[[
    "ticker", "name", "poids_%",
    "nb_actions", "prix_ZAR", "montant_ZAR"
]]

display_df.columns = [
    "Ticker", "Entreprise", "Poids (%)",
    "Nb actions", "Prix (ZAR)", "Montant (ZAR)"
]

print("Allocation finale du portefeuille JSE\n")
print(display_df.to_string(index=False))
print(f"\nCapital total investi : {display_df['Montant (ZAR)'].sum():,.2f} ZAR")

Allocation finale du portefeuille JSE

Ticker      Entreprise  Poids (%)  Nb actions  Prix (ZAR)  Montant (ZAR)
GFI.JO     Gold Fields     40.000           1   27754.490      27754.490
IMP.JO Impala Platinum     27.400           3    9126.200      27378.590
REM.JO          Remgro     13.800           1   16239.750      16239.750
SBK.JO   Standard Bank      5.000           1   20781.220      20781.220
FSR.JO       FirstRand      5.000           1    7342.260       7342.260

Capital total investi : 99,496.31 ZAR


In [8]:
# Evolution de la valeur du portefeuille si on avait
# investi selon cette allocation depuis 2019
weights_dict = dict(zip(
    allocation_df["ticker"],
    allocation_df["poids_"] / 100
    if "poids_" in allocation_df.columns
    else allocation_df["poids_%"] / 100
))

prices_port = prices[list(weights_dict.keys())].dropna()

# Rendement journalier pondéré
daily_returns = prices_port.pct_change().dropna()
weighted_returns = daily_returns.multiply(
    pd.Series(weights_dict), axis=1
).sum(axis=1)

# Valeur cumulée
portfolio_value = 100_000 * (1 + weighted_returns).cumprod()

# Benchmark
benchmark_prices = prices.mean(axis=1)
benchmark_value  = 100_000 * (
    1 + benchmark_prices.pct_change().dropna()
).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=portfolio_value.index,
    y=portfolio_value.values,
    mode="lines",
    name="Portefeuille optimisé",
    line=dict(color="#1D9E75", width=2.5)
))

fig.add_trace(go.Scatter(
    x=benchmark_value.index,
    y=benchmark_value.values,
    mode="lines",
    name="Benchmark (moyenne JSE)",
    line=dict(color="#D85A30", width=2, dash="dash")
))

fig.add_hline(
    y=100_000,
    line_dash="dot",
    line_color="gray",
    annotation_text="Capital initial"
)

fig.update_layout(
    title="Evolution du portefeuille optimisé vs Benchmark (2019-2024)",
    xaxis_title="Date",
    yaxis_title="Valeur du portefeuille (ZAR)",
    template="plotly_white",
    width=850, height=500,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

fig.write_html("../results/figures/05_evolution_portefeuille.html")
fig.write_image("../results/figures/05_evolution_portefeuille.png")

## Conclusion du pipeline

On a construit un pipeline complet de A à Z :

**1. Collecte** — 22 actions JSE, 5 ans de prix historiques et fondamentaux financiers via yfinance.

**2. Preprocessing** — création du label de surperformance vs l'indice JSE All Share, ajout de features dynamiques (momentum, volatilité, force relative).

**3. Modèle ML** — Gradient Boosting entraîné sur 2019-2021, testé sur 2022. AUC de 0.46 — modeste mais honnête pour un marché émergent avec des données limitées.

**4. Backtesting** — la stratégie ML génère +6.5% vs +1.3% pour le Buy & Hold sur 2022, soit un avantage de +5.2%.

**5. Optimisation** — Markowitz répartit le capital de façon optimale sur les actions sélectionnées par le modèle, en maximisant le ratio de Sharpe.

### Pistes d'amélioration futures
- Données fondamentales trimestrielles historiques
- Élargir l'univers à toutes les actions JSE (~350 titres)
- Ajouter des données macro (taux ZAR, prix des matières premières)
- Tester des modèles LSTM pour capturer les séquences temporelles
- Déployer un dashboard Streamlit interactif